In [2]:
!pip install transformers datasets torch accelerate rouge_score evaluate -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [3]:
import os
import torch
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configuration Paths
# Ensure your file is exactly at this path in Drive
PROJECT_FOLDER = "/content/drive/MyDrive/urdu_gpt2_headline_project"
DATA_FILE = f"{PROJECT_FOLDER}/smart_augmented_CLEANED.ndjson"
OUTPUT_DIR = f"{PROJECT_FOLDER}/gpt2_deep_model"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Training Data: {DATA_FILE}")
print(f"Model Output: {OUTPUT_DIR}")

Mounted at /content/drive
Training Data: /content/drive/MyDrive/urdu_gpt2_headline_project/smart_augmented_CLEANED.ndjson
Model Output: /content/drive/MyDrive/urdu_gpt2_headline_project/gpt2_deep_model


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Imran1/gpt2-urdu-news"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Fix padding for GPT-2
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

Loading Imran1/gpt2-urdu-news...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/470 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/909 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/663M [00:00<?, ?B/s]

In [7]:
import random
from datasets import load_dataset

# 1. Load the Dataset
print("Loading dataset...")
dataset = load_dataset("json", data_files=DATA_FILE, split="train")
# Split 90% Training / 10% Validation
dataset = dataset.train_test_split(test_size=0.1)

SEPARATOR = "\n\n###\n\n"

def preprocess_deep_training(examples):
    inputs = []
    labels = []

    for article, headline in zip(examples["text"], examples["headline"]):
        if not article or not headline:
            continue

        # --- DYNAMIC AUGMENTATION: LEAD DROPPING ---
        # 20% chance to drop the first sentence during training.
        # This prevents the model from just memorizing the start of the text.
        if random.random() < 0.20:
            sentences = article.split("۔")
            if len(sentences) > 2:
                # Keep everything after the first full stop
                article = "۔".join(sentences[1:])
        # -------------------------------------------

        # 1. Format Input (Professor's Strategy)
        # Add pipe symbol to break up sentences
        article_text = article.replace("۔", "۔ | ") + SEPARATOR
        headline_text = headline + tokenizer.eos_token

        # 2. Tokenize separately
        article_tokens = tokenizer(article_text, add_special_tokens=False)['input_ids']
        headline_tokens = tokenizer(headline_text, add_special_tokens=False)['input_ids']

        # 3. Combine
        full_input_ids = article_tokens + headline_tokens

        # 4. Create Masked Labels (-100 on Article)
        full_labels = [-100] * len(article_tokens) + headline_tokens

        # 5. Truncate if necessary (Cut from start of article)
        if len(full_input_ids) > 1024:
            diff = len(full_input_ids) - 1024
            full_input_ids = full_input_ids[diff:]
            full_labels = full_labels[diff:]

        inputs.append(full_input_ids)
        labels.append(full_labels)

    return {"input_ids": inputs, "labels": labels}

# Apply processing
print("Processing data...")
tokenized_train = dataset["train"].map(preprocess_deep_training, batched=True, remove_columns=dataset["train"].column_names)
tokenized_eval = dataset["test"].map(preprocess_deep_training, batched=True, remove_columns=dataset["test"].column_names)
print("Data processing complete.")

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Processing data...


Map:   0%|          | 0/33426 [00:00<?, ? examples/s]

Map:   0%|          | 0/3714 [00:00<?, ? examples/s]

Data processing complete.


In [5]:
# ==============================================================================
# SAFE A100 CONFIGURATION (Memory Optimized)
# ==============================================================================

from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification
import os

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Use DataCollatorForTokenClassification for -100 masking support
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=5,
    learning_rate=3e-5,

    # --- MEMORY FIX ---
    # We reduce the physical batch size but keep the math the same.
    # Old: Batch 32 / Accum 1 = 32
    # New: Batch 8  / Accum 4 = 32 (Same speed, 1/4th memory usage)
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    # ------------------

    bf16=True,                       # Keep BF16 for A100 speed
    fp16=False,

    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    eval_strategy="steps",
    eval_steps=250,

    logging_steps=50,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)



print(" Resuming Training ...")

# Check if there are existing checkpoints
import glob
checkpoints = glob.glob(f"{OUTPUT_DIR}/checkpoint-*")

if len(checkpoints) > 0:
    print(f"Found {len(checkpoints)} checkpoints. Resuming from the latest one.")
    # This automatically loads the weights AND the optimizer state
    trainer.train(resume_from_checkpoint=True)
else:
    print(" No checkpoints found! Starting from scratch.")
    trainer.train()

 Resuming Training ...
Found 2 checkpoints. Resuming from the latest one.


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
1500,2.844300,2.714240
1750,2.753200,2.646234
2000,2.714500,2.580007
2250,2.527000,2.529360
2500,2.490500,2.480012
2750,2.477200,2.447941
3000,2.455700,2.400782
3250,2.307200,2.376569
3500,2.289100,2.342648
3750,2.259400,2.319622


In [8]:
final_path = f"{OUTPUT_DIR}/final_model"
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)
print(f"Success! Final model saved to: {final_path}")

Success! Final model saved to: /content/drive/MyDrive/urdu_gpt2_headline_project/gpt2_deep_model/final_model


In [10]:
!pip install bert_score sacrebleu rouge_score evaluate -q

import evaluate
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# --- CONFIGURATION ---
MODEL_PATH = "/content/drive/MyDrive/urdu_gpt2_headline_project/gpt2_deep_model/final_model"
DATA_FILE = "/content/drive/MyDrive/urdu_gpt2_headline_project/smart_augmented_CLEANED.ndjson"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Resources
print("Loading resources...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(DEVICE)
dataset = load_dataset("json", data_files=DATA_FILE, split="train").train_test_split(test_size=0.1, seed=42)["test"]

# Select 200 samples for robust testing
test_dataset = dataset.select(range(20))

# 2. Metrics Load
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")


Loading resources...


In [1]:
import torch
import numpy as np
from tqdm import tqdm
import evaluate

# 1. Setup Metrics
bertscore = evaluate.load("bertscore")
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# 2. Generation Loop
print(f"Generating headlines for {len(test_dataset)} samples...")
predictions = []
references = []

for item in tqdm(test_dataset):

    # The prompt format
    prompt = item['text'] + "\n\n###\n\n"

    inputs = tokenizer(prompt, return_tensors="pt", max_length=900, truncation=True).to(DEVICE)

    # --- Generating ---
    output_ids = model.generate(
        **inputs,
        max_new_tokens=35,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=2,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # --- Cleaning ---
    if "###" in decoded:
        pred = decoded.split("###")[-1].strip()
    else:
        pred = decoded.replace(prompt, "").strip()

    predictions.append(pred)
    references.append(item['headline'])

# 3. Visual Inspection
print("\n" + "="*40)
print("      VISUAL INSPECTION (First 5)")
print("="*40)
for i in range(min(5, len(predictions))):
    print(f"\n--- Example {i+1} ---")
    print(f"Generated: {predictions[i]}")
    print(f"Actual:    {references[i]}")
print("="*40)

# 4. Metrics Calculation
print("\n--- Calculating Metrics ---")

# A. BERTScore (Updated to include Precision & Recall)
bert_res = bertscore.compute(predictions=predictions, references=references, lang="ur")
print(f"BERT Precision: {np.mean(bert_res['precision'])*100:.2f}")
print(f"BERT Recall:    {np.mean(bert_res['recall'])*100:.2f}")
print(f"BERT F1:        {np.mean(bert_res['f1'])*100:.2f}")

# B. ROUGE
rouge_res = rouge.compute(predictions=predictions, references=references)
print(f"ROUGE-1:      {rouge_res['rouge1']*100:.2f}")
print(f"ROUGE-2:      {rouge_res['rouge2']*100:.2f}")
print(f"ROUGE-L:      {rouge_res['rougeL']*100:.2f}")

# C. BLEU
formatted_refs = [[ref] for ref in references]
bleu_res = bleu.compute(predictions=predictions, references=formatted_refs)
print(f"BLEU:         {bleu_res['bleu']*100:.2f}")

ModuleNotFoundError: No module named 'evaluate'

In [5]:
# 1. Install Dependencies
!pip install transformers datasets torch accelerate rouge_score bert_score sacrebleu evaluate -q

import os
import torch
import re
import numpy as np
from tqdm import tqdm
import evaluate
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# 2. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 3. Configuration
PROJECT_FOLDER = "/content/drive/MyDrive/urdu_gpt2_headline_project"
DATA_FILE = f"{PROJECT_FOLDER}/smart_augmented_CLEANED.ndjson"
MODEL_PATH = f"{PROJECT_FOLDER}/gpt2_deep_model/final_model"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(DEVICE)

# 4. Load Test Data
dataset = load_dataset("json", data_files=DATA_FILE, split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = dataset["test"].select(range(50)) # Testing on 50 samples

# 5. Load Metrics
bertscore = evaluate.load("bertscore")
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# --- FIX: URDU NORMALIZATION FUNCTION ---
def normalize_urdu(text):
    # Remove diacritics (Zer, Zabar, Pesh, etc.)
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)
    # Normalize Alifs
    text = re.sub(r'[آأ]', 'ا', text)
    # Normalize Yeh
    text = re.sub(r'[يى]', 'ی', text)
    # Remove punctuation for better word matching
    text = re.sub(r'[۔،:؛\?!]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 6. Generation Loop
print(f"Generating headlines...")
predictions = []
references = []

for item in tqdm(test_dataset):
    prompt = item['text'] + "\n\n###\n\n"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=900, truncation=True).to(DEVICE)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=35,
        num_beams=5,
        early_stopping=True,
        no_repeat_ngram_size=2,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if "###" in decoded:
        pred = decoded.split("###")[-1].strip()
    else:
        pred = decoded.replace(prompt, "").strip()

    predictions.append(pred)
    references.append(item['headline'])

# 7. Apply Normalization & Tokenization
print("Normalizing and Tokenizing...")
norm_preds = [normalize_urdu(p) for p in predictions]
norm_refs = [normalize_urdu(r) for r in references]

# Tokenize using model's tokenizer (The "Space" Fix)
pred_tok = [" ".join(tokenizer.tokenize(p)) for p in norm_preds]
ref_tok = [" ".join(tokenizer.tokenize(r)) for r in norm_refs]

# 8. Metrics Calculation
print("\n--- Calculating Normalized Metrics ---")

# BERTScore (Semantics)
bert_res = bertscore.compute(predictions=predictions, references=references, lang="ur")
print(f"BERT F1:        {np.mean(bert_res['f1'])*100:.2f}")

# ROUGE (Exact Match - Normalized)
rouge_res = rouge.compute(predictions=pred_tok, references=ref_tok)
print(f"ROUGE-L:      {rouge_res['rougeL']*100:.2f}")

# BLEU (Exact Match - Normalized)
formatted_refs = [[r] for r in ref_tok]
bleu_res = bleu.compute(predictions=pred_tok, references=formatted_refs)
print(f"BLEU:         {bleu_res['bleu']*100:.2f}")

Loading model from: /content/drive/MyDrive/urdu_gpt2_headline_project/gpt2_deep_model/final_model
Generating headlines...


100%|██████████| 50/50 [00:25<00:00,  1.98it/s]


Normalizing and Tokenizing...

--- Calculating Normalized Metrics ---
BERT F1:        68.37
ROUGE-L:      1.42
BLEU:         2.13
